# Лабораторная работа №1. Первая маршрутизируемая сеть

Соберём сеть из двух компьютеров и двух маршрутизаторов и разберём один пакет от начала до конца.

Здесь теория идёт вместе с практикой: прочитали короткий фрагмент → сразу посмотрели это на реальном стенде.

**Работаем в двух окнах:** notebook и терминалы `pc1`, `r1`, `r2`, `pc2`.  
Терминалы `r1` и `r2` сразу открываются в FRRouting CLI (`vtysh`).

После некоторых вопросов есть необязательный prompt для ИИ. Используйте его только после собственного ответа: он должен проверить рассуждение, но не выдавать решение.

## Схема стенда

![Схема стенда](assets/topology.svg)

Адреса пока не настроены. К концу работы `pc1` и `pc2` должны обмениваться ICMP через оба маршрутизатора.

| Узел | Интерфейс | Адрес |
|---|---|---|
| pc1 | eth1 | `192.168.10.10/24` |
| r1 | eth1 | `192.168.10.1/24` |
| r1 | eth2 | `10.0.12.1/30` |
| r2 | eth1 | `10.0.12.2/30` |
| r2 | eth2 | `192.168.20.1/24` |
| pc2 | eth1 | `192.168.20.10/24` |

## 1. Сначала посмотрим, что уже есть

На `pc1`:

```bash
ip -br link
ip -br addr
ip -4 route
```

На `r1`:

```text
show interface
show ip route
```

Линки уже существуют, но нужной IP-конфигурации ещё нет. Физическое соединение и IP-маршрутизация — разные вещи.

### Q1. `pc1` и `r1` подключены к сети, но выполняют разные роли. Что делает оконечный узел, а что делает маршрутизатор?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

## Что означает /24

![Что означает /24](assets/prefix24.svg)

`/24` означает: первые 24 бита адреса описывают сеть.

Для `192.168.10.10/24` локальная сеть — `192.168.10.0/24`.  
Адрес `192.168.20.10` в неё уже не входит.

## 2. Настроим адреса

> `eth0` — служебный management-интерфейс C9s и сохраняет свой default route. Мы его не трогаем. Для удалённой лабораторной LAN добавляем более специфичный `/24` маршрут через `eth1`: по longest-prefix match он будет выбран раньше default `/0`.

На `pc1`:

```bash
ip addr add 192.168.10.10/24 dev eth1
ip route replace 192.168.20.0/24 via 192.168.10.1 dev eth1
ip -4 route
```

На `pc2`:

```bash
ip addr add 192.168.20.10/24 dev eth1
ip route replace 192.168.10.0/24 via 192.168.20.1 dev eth1
ip -4 route
```

Теперь маршрутизаторы.

На `r1`:

```text
configure terminal
interface eth1
 ip address 192.168.10.1/24
 exit
interface eth2
 ip address 10.0.12.1/30
 exit
end
show ip route
```

На `r2`:

```text
configure terminal
interface eth1
 ip address 10.0.12.2/30
 exit
interface eth2
 ip address 192.168.20.1/24
 exit
end
show ip route
```

**Обратите внимание:** connected routes появились сами. Мы не вводили для них `ip route ...`.

### Q2. Почему после назначения адреса интерфейсу маршрут к непосредственно подключённой сети появляется автоматически?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

### Q3. `pc1` хочет отправить пакет на `192.168.20.10`. Этот адрес не входит в его локальную `/24`. Куда `pc1` отправит пакет первым?

`введите ваш ответ`

Проверьте решение Linux на `pc1`:

```bash
ip route get 192.168.20.10
```

Вы должны увидеть, что для `192.168.20.10` выбран более специфичный маршрут `192.168.20.0/24` через `192.168.10.1` по `eth1`, а не служебный management default.

## Проверка 1

Запустите **CHECK**. Сейчас должны быть настроены адреса, `/24` маршруты хостов к удалённым LAN и connected routes маршрутизаторов.

## 3. Попробуем пройти всю сеть

Интерфейсы уже настроены, R1 видит R2 по сети `10.0.12.0/30`.

Но мы ещё не добавляли маршруты к удалённым LAN.

### Q4. До запуска `ping`: дойдёт ли пакет `pc1 → pc2` до конца? Если нет — на каком маршрутизаторе вы ожидаете остановку?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

Теперь проверьте гипотезу.

На `pc1`:

```bash
ping -c 2 192.168.20.10
traceroute 192.168.20.10
```

Сравните результаты: `ping` отвечает на вопрос «есть ли end-to-end связность», а `traceroute` помогает увидеть, **на каком L3-hop путь перестаёт продолжаться**. Сейчас первый hop должен быть R1 (`192.168.10.1`), а дальше маршрут ещё не построен полностью.

На `r1`:

```text
show ip route
show ip route 192.168.20.10
```

R1 знает свою LAN и линк R1–R2, но пока не знает, что `192.168.20.0/24` находится за R2.

## Где пакет останавливается

![Где пакет останавливается](assets/route-missing.svg)

Физический линк R1–R2 есть. Проблема не в кабеле и не в состоянии интерфейса — у R1 нет маршрутной информации для LAN-B.

## 4. Добавим недостающую информацию

На `r1`:

```text
configure terminal
ip route 192.168.20.0/24 10.0.12.2
end
show ip route
```

На `r2`:

```text
configure terminal
ip route 192.168.10.0/24 10.0.12.1
end
show ip route
```

Теперь снова на `pc1`:

```bash
ping -c 3 192.168.20.10
traceroute 192.168.20.10
```

После настройки ожидаемый маршрут:

```text
192.168.10.1  →  10.0.12.2  →  192.168.20.10
R1               R2             pc2
```

`traceroute` нужен здесь как диагностика пути, а `ping` остаётся проверкой end-to-end связности.

### Q5. Что означает next hop `10.0.12.2` в маршруте R1 к `192.168.20.0/24`? Почему это адрес R2, а не адрес `pc2`?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q5, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

## Проверка 2

Запустите **CHECK**.

Теперь checker должен увидеть:
- удалённый маршрут на R1;
- обратный маршрут на R2;
- `pc1 → pc2`;
- `pc2 → pc1`.

Если прямой маршрут есть, а обратного нет, Echo Request может уйти вперёд, но Echo Reply не сможет вернуться.

## 5. Что меняется по пути

Перед просмотром анимации сделайте предсказание.

### Q6. Когда пакет проходит R1 и R2, меняется ли конечный IPv4 destination `192.168.20.10`? А что должно меняться на каждом L2-сегменте?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q6, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

## Один IP-пакет, три Ethernet-сегмента

![Один IP-пакет, три Ethernet-сегмента](assets/route-success.svg)

IP destination остаётся адресом `pc2`. На каждом сегменте меняется L2 next-hop, а каждый маршрутизатор уменьшает TTL.

## Посмотрим реальный пакет

Для разбора пакетов используем встроенный viewer CMS Labs, а не ручной `tcpdump`.

1. Откройте терминал `pc1`.
2. Подготовьте команду:

```bash
ping -c 3 192.168.20.10
```

3. Запустите следующую ячейку.
4. Пока идёт захват, выполните `ping` в терминале `pc1`.

После завершения появится таблица пакетов. Для любого кадра можно открыть protocol tree, hex dump и raw JSON.


In [ ]:
%%capture_traffic pc1:eth1 --filter "arp or icmp" --timeout 12 --packets 100 --save captures/pc1-icmp.pcap
import time
print("Запустите сейчас ping -c 3 192.168.20.10 в терминале pc1")
time.sleep(10)


Теперь посмотрим тот же обмен **после R1**, на транзитном интерфейсе `r1:eth2`.

Запустите следующую ячейку и снова выполните `ping` из `pc1`. Сравните Ethernet source/destination и TTL с первым capture.


In [ ]:
%%capture_traffic r1:eth2 --filter "icmp" --timeout 12 --packets 100 --save captures/r1-transit-icmp.pcap
import time
print("Повторите ping -c 3 192.168.20.10 в терминале pc1")
time.sleep(10)


### Статический результат capture

Интерактивный viewer удобен для разбора пакетов, но ниже мы сохраняем ещё и обычный вывод из тех же `.pcap`.

`DataFrame` и текстовый protocol tree остаются в output ячейки. Поэтому после выполнения и сохранения notebook преподаватель увидит реальные результаты захвата и в HTML/PDF-экспорте, даже без интерактивных widgets.


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

captures = {
    "LAN-A · pc1:eth1": PcapViewer("captures/pc1-icmp.pcap", "arp or icmp", limit=30),
    "Transit · r1:eth2": PcapViewer("captures/r1-transit-icmp.pcap", "icmp", limit=30),
}

frames = {}
for title, viewer in captures.items():
    print(f"\n{title}")
    frame = viewer.packets()
    frames[title] = frame
    display(frame)

# Покажем компактный Wireshark/tshark protocol tree первого ICMP-кадра
# из capture на pc1. Это статический текст и он тоже попадает в HTML/PDF.
lan = frames["LAN-A · pc1:eth1"]
icmp = lan[lan["Protocol"].astype(str).str.contains("ICMP", case=False, na=False)]
if not icmp.empty:
    frame_number = int(icmp.iloc[0]["№"])
    tree = captures["LAN-A · pc1:eth1"].packet(frame_number, "protocols")
    print(f"\nProtocol tree · frame {frame_number}\n")
    print("\n".join(tree.splitlines()[:80]))
else:
    print("ICMP-кадр не найден: повторите capture и ping.")


## 6. Сломаем только один маршрут

На `r1`:

```text
configure terminal
no ip route 192.168.20.0/24 10.0.12.2
end
show ip route
```

Не восстанавливайте его сразу.

На `pc1` проверьте:

```bash
ping -c 1 192.168.10.1
ping -c 1 10.0.12.2
ping -c 1 192.168.20.10
```

Идея здесь простая: не спрашивать «работает сеть или нет», а выяснить **до какого уровня она работает**.

### Q7. Какие из трёх ping продолжают работать? Что это позволяет исключить из возможных причин неисправности?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q7, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

Верните маршрут на `r1`:

```text
configure terminal
ip route 192.168.20.0/24 10.0.12.2
end
```

Повторите end-to-end ping и запустите финальный **CHECK**.

## 7. Соберите всё в одну модель

Не перечисляйте команды. Опишите, что происходит с одним Echo Request от момента, когда `pc1` получает адрес назначения `192.168.20.10`, до доставки на `pc2`.

Достаточно 5–8 предложений.

### Q8. Опишите путь пакета: решение `pc1` по таблице маршрутизации, gateway R1, next hop R2, R2 и доставка в LAN-B. Укажите, что происходит с L2 destination и TTL.

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q8, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду настройки.

</details>

## Что нужно унести из этой работы

- адрес интерфейса создаёт connected route к локальной сети;
- для удалённой лабораторной сети host использует более специфичный маршрут через gateway; такой `/24` выигрывает у служебного default `/0` по longest-prefix match;
- маршрутизатор пересылает пакет только если знает подходящий маршрут;
- next hop — следующий L3-сосед, а не обязательно конечный host;
- для двусторонней связи нужен обратный путь;
- IP-пакет живёт end-to-end, а Ethernet-заголовок относится к конкретному сегменту;
- troubleshooting удобнее делать по пути пакета, а не случайным набором команд.